 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison 
Name           : Martin Law                         <br>
Student Number : L00203482                          <br>
Due Date       : Tuesday 12th May 2026              <br>
Assignment     : CA2                                <br>
Module         : AI for Vision and NLP              <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level

This application will analyse scanned or photographed document images using a combined computer vision, OCR, NLP, and reporting pipeline.

The planned high level process is:

1. load scanned or photographed document images
2. preprocess images using OpenCV
3. extract text using Tesseract OCR
4. compare OCR results across different preprocessed image versions
5. clean and process extracted text
6. extract NLP features such as tokens, stems, lemmas, keywords, and named entities
7. detect visual document features such as text blocks, tables, signatures, diagrams, figures, and form-like regions
8. combine the text and visual findings
9. export structured document summaries, annotated images, CSV files, and JSON reports

At this development stage, the notebook loads and displays the input document images. OCR, NLP processing, image preprocessing, segmentation, and feature detection will be added in later commits

In [ ]:
# high-level pipeline placeholder for the first development stage

pipeline_steps = [
    "load scanned or photographed document images",
    "preprocess images using OpenCV",
    "extract text using Tesseract OCR",
    "compare OCR confidence across preprocessing methods",
    "clean and process extracted text",
    "extract NLP features",
    "detect visual document features",
    "combine text and image findings",
    "export structured document output"
]

for step_number, step_description in enumerate(pipeline_steps, start=1):
    print(f"{step_number}. {step_description}")

# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

In [ ]:
# pip installs
# run this cell only if the required packages are not already instlal ed in the selected environment

# %pip install opencv-python pytesseract pandas numpy matplotlib scikit-learn nltk jupyter ipykernel

## Imports

In [ ]:
# imports

from pathlib import Path
import platform
import json
import shutil

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytesseract

# Support Functions

In [ ]:
# support functions and project folders

PROJECT_DIR = Path.cwd()
INPUT_DIR = PROJECT_DIR / "data" / "input_documents"
OUTPUT_DIR = PROJECT_DIR / "outputs"

PREPROCESSED_DIR = OUTPUT_DIR / "preprocessed_images"
OCR_DIR = OUTPUT_DIR / "ocr_results"
OCR_TEXT_DIR = OUTPUT_DIR / "ocr_text"
ANNOTATED_DIR = OUTPUT_DIR / "annotated_images"
JSON_DIR = OUTPUT_DIR / "json_reports"
CSV_DIR = OUTPUT_DIR / "csv_reports"

def create_project_folders():
    """ create the folder structure used by this notebook"""
    folders = [
        INPUT_DIR,
        OUTPUT_DIR,
        PREPROCESSED_DIR,
        OCR_DIR,
        OCR_TEXT_DIR,
        ANNOTATED_DIR,
        JSON_DIR,
        CSV_DIR
    ]

    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)

    return folders

def list_input_documents():
    """return supported image files from the input document folder"""
    supported_extensions = [
        "*.jpg",
        "*.jpeg",
        "*.png",
        "*.bmp",
        "*.tif",
        "*.tiff"    
    ]

    image_paths = []

    for extension in supported_extensions:
        image_paths.extend(INPUT_DIR.glob(extension))

    return sorted(image_paths)

created_folders = create_project_folders()
input_documents = list_input_documents()

print("Project directory:", PROJECT_DIR)
print("Operating system:", platform.system())
print("Input directory:", INPUT_DIR)
print("Output directory:", OUTPUT_DIR)
print("Preprocessed image directory:", PREPROCESSED_DIR)
print("OCR results directory:", OCR_DIR)
print("Number of input document images found:", len(input_documents))

for image_path in input_documents:
    print("-", image_path.name)

# NLP

## Stage 5 NLP Plan

this section will later contain the OCR text cleaning and NLP processing.

Planned NLP functionality:

1. clean extracted OCR text
2. tokenise the text
3. remove stopwords
4. apply stemming
5. apply lemmatisation
6. extract keywords using TF-IDF
7. extract simple named entities such as dates, receipt numbers, IDs, names, and document references (if there)
8. identify references to tables, figures, diagrams, and signatures

there will be no NLP processing implemented in this commit

In [ ]:
# NLP development status for this commit

nlp_development_status = {
    "ocr_text_available": "implemented in Stage 4 below",
    "ocr_text_cleaning": "planned",
    "tokenisation": "planned",
    "stopword_removal": "planned",
    "stemming": "planned",
    "lemmatisation": "planned",
    "tfidf_keyword_extraction": "planned",
    "entity_extraction": "planned",
    "named_entity_extraction": "planned",
    "figure_table_reference_detection": "planned"
}

nlp_development_status

# Vision

## Stage 3: Image Preprocessing and Enhancement

This section prepares scanned or photographed document images for later OCR and feature detection.

The preprocessing steps are:

1. load the document image
2. convert it to grayscale
3. reduce noise using median blur
4. enhance contrast using CLAHE
5. apply Otsu thresholding
6. apply adaptive thresholding
7. apply a cleaner adaptive threshold for OCR comparison
8. detect edges using Canny edge detection
9. save the processed images for later stages

The OCR phase below will compare several of these preprocessed outputs and select the one with the best tereract confidence score

In [ ]:
# load and display input document images

input_documents = list_input_documents()

if len(input_documents) == 0:
    raise FileNotFoundError(
        f"No document images were found in {INPUT_DIR}. "
        "Add .jpg, .jpeg, .png, .bmp, .tif, or .tiff files before running this cell"
    )

loaded_documents = []

for image_path in input_documents:
    image = cv2.imread(str(image_path))

    if image is None:
        print(f"Could not load image: {image_path.name}")
        continue

    loaded_documents.append({
        "path": image_path,
        "filename": image_path.name,
        "image": image,
        "height": image.shape[0],
        "width": image.shape[1],
        "channels": image.shape[2] if len(image.shape) == 3 else 1
    })

print("Loaded document images:")

for document in loaded_documents:
    print(
        f"- {document['filename']} "
        f"({document['width']} x {document['height']}, "
        f"{document['channels']} channel(s))"
    )

In [ ]:
# display a sample of the loaded document images

documents_to_display = loaded_documents[:4]

plt.figure(figsize=(14, 10))

for index, document in enumerate(documents_to_display, start=1):
    image_rgb = cv2.cvtColor(document["image"], cv2.COLOR_BGR2RGB)

    plt.subplot(2, 2, index)
    plt.imshow(image_rgb)
    plt.axis("off")
    plt.title(document["filename"])

plt.tight_layout()
plt.show()

In [ ]:
# OpenCV preprocessing functions

def preprocess_document_image(image):
    """Apply basic OpenCV preprocessing steps to a document image."""
    if image is None:
        raise ValueError("Cannot preprocess an empty image.")

    # convert from colour to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # reduce small noise while keeping text edges reasonably sharp
    denoised = cv2.medianBlur(gray, 3)

    # improve local contrast, useful for photographed documents with uneven lighting
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    enhanced = clahe.apply(denoised)

    # global thresholding using Otsu's method
    _, otsu_threshold = cv2.threshold(
        enhanced,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # adaptive thresholding handles uneven lighting better than a single global threshold
    adaptive_threshold = cv2.adaptiveThreshold(
        enhanced,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        15
    )

    # a softer adaptive threshold version for OCR comparison
    adaptive_threshold_clean = cv2.adaptiveThreshold(
        enhanced,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        51,
        11
    )

    # remove small isolated black noise from the thresholded image
    noise_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))

    adaptive_threshold_clean = cv2.morphologyEx(
        adaptive_threshold_clean,
        cv2.MORPH_OPEN,
        noise_kernel
    )

    # edge detection is useful later for detecting boxes, tables, and page boundaries
    edges = cv2.Canny(
        enhanced,
        threshold1=50,
        threshold2=150
    )

    return {
        "gray": gray,
        "denoised": denoised,
        "enhanced": enhanced,
        "otsu_threshold": otsu_threshold,
        "adaptive_threshold": adaptive_threshold,
        "adaptive_threshold_clean": adaptive_threshold_clean,
        "edges": edges
    }


In [ ]:
# preprocess all loaded document images and save the results

if "loaded_documents" not in globals():
    raise NameError("loaded_documents does not exist. Run the image loading cell before this cell.")

if len(loaded_documents) == 0:
    raise ValueError("No loaded documents are available for preprocessing.")

preprocessed_documents = []

for document in loaded_documents:
    filename_stem = Path(document["filename"]).stem

    processed_images = preprocess_document_image(document["image"])

    output_paths = {}

    for processing_name, processed_image in processed_images.items():
        output_path = PREPROCESSED_DIR / f"{filename_stem}_{processing_name}.png"
        cv2.imwrite(str(output_path), processed_image)
        output_paths[processing_name] = output_path

    preprocessed_documents.append({
        "filename": document["filename"],
        "original_image": document["image"],
        "processed_images": processed_images,
        "output_paths": output_paths
    })

print("Preprocessed document images saved:")

for document in preprocessed_documents:
    print(f"- {document['filename']}")

    for processing_name, output_path in document["output_paths"].items():
        print(f"  {processing_name}: {output_path.name}")


In [ ]:
# display preprocessing results for the first document

example_document = preprocessed_documents[0]

original_rgb = cv2.cvtColor(example_document["original_image"], cv2.COLOR_BGR2RGB)

display_images = [
    ("Original", original_rgb, None),
    ("Grayscale", example_document["processed_images"]["gray"], "gray"),
    ("Denoised", example_document["processed_images"]["denoised"], "gray"),
    ("Enhanced", example_document["processed_images"]["enhanced"], "gray"),
    ("Otsu Threshold", example_document["processed_images"]["otsu_threshold"], "gray"),
    ("Adaptive Threshold", example_document["processed_images"]["adaptive_threshold"], "gray"),
    ("Clean Adaptive Threshold", example_document["processed_images"]["adaptive_threshold_clean"], "gray"),
    ("Canny Edges", example_document["processed_images"]["edges"], "gray")
]

plt.figure(figsize=(16, 12))

for index, (title, image, colour_map) in enumerate(display_images, start=1):
    plt.subplot(3, 3, index)
    plt.imshow(image, cmap=colour_map)
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

print("Displayed preprocessing results for:", example_document["filename"])


## Stage 4: Tesseract OCR Extraction and Confidence Comparison

This stage extracts text from each document image using Tesseract OCR.

To avoid assuming that one preprocessing method is always best, OCR is tested on several image versions:

1. grayscale
2. denoised
3. contrast-enhanced
4. clean adaptive threshold

The notebook then compares the average Tesseract confidence score and selects the best OCR result for each document.

This stage addresses the OCR accuracy part of the rubric. NLP preprocessing is added in the next phase.

In [ ]:
# Tesseract OCR configuration

OCR_CONFIG = "--oem 3 --psm 6"

def configure_tesseract_for_current_system():
    """configure the Tesseract executable path where needed"""
    system_name = platform.system()

    if system_name == "Darwin":
        possible_paths = [
            "/opt/homebrew/bin/tesseract",
            "/usr/local/bin/tesseract"
        ]

        for possible_path in possible_paths:
            if Path(possible_path).exists():
                pytesseract.pytesseract.tesseract_cmd = possible_path
                break

    detected_path = shutil.which("tesseract") or pytesseract.pytesseract.tesseract_cmd

    try:
        version = pytesseract.get_tesseract_version()
        print("Tesseract detected:", version)
        print("Tesseract path:", detected_path)
    except Exception as error:
        raise RuntimeError(
            "Tesseract could not be found by pytesseract. "
            "On macOS, install it with: brew install tesseract"
        ) from error


configure_tesseract_for_current_system()

In [ ]:
# OCR helper functions

def normalise_ocr_dataframe(ocr_dataframe):
    """clean the dataframe returned by pytesseract.image_to_data"""
    cleaned_dataframe = ocr_dataframe.copy()

    cleaned_dataframe["text"] = cleaned_dataframe["text"].fillna("").astype(str)
    cleaned_dataframe["text"] = cleaned_dataframe["text"].str.strip()
    cleaned_dataframe["conf"] = pd.to_numeric(cleaned_dataframe["conf"], errors="coerce")

    cleaned_dataframe = cleaned_dataframe[
        (cleaned_dataframe["text"] != "") &
        (cleaned_dataframe["conf"].notna()) &
        (cleaned_dataframe["conf"] >= 0)
    ].copy()

    return cleaned_dataframe


def run_ocr_on_image(image):
    """run Tesseract OCR on one image and return text, word data, and confidence metrics"""
    ocr_dataframe = pytesseract.image_to_data(
        image,
        config=OCR_CONFIG,
        output_type=pytesseract.Output.DATAFRAME
    )

    ocr_dataframe = normalise_ocr_dataframe(ocr_dataframe)

    extracted_text = " ".join(ocr_dataframe["text"].tolist())

    if len(ocr_dataframe) == 0:
        average_confidence = 0.0
        median_confidence = 0.0
    else:
        average_confidence = float(ocr_dataframe["conf"].mean())
        median_confidence = float(ocr_dataframe["conf"].median())

    return {
        "text": extracted_text,
        "ocr_dataframe": ocr_dataframe,
        "average_confidence": average_confidence,
        "median_confidence": median_confidence,
        "word_count": int(len(ocr_dataframe)),
        "character_count": int(len(extracted_text))
    }


def select_ocr_candidate_images(preprocessed_document):
    """select the preprocessed image versions used for OCR comparison"""
    processed_images = preprocessed_document["processed_images"]

    candidate_names = [
        "gray",
        "denoised",
        "enhanced",
        "adaptive_threshold_clean"
    ]

    candidates = {}

    for candidate_name in candidate_names:
        if candidate_name in processed_images:
            candidates[candidate_name] = processed_images[candidate_name]

    return candidates

In [ ]:
# run OCR on each selected preprocessing version and select the best result

if "preprocessed_documents" not in globals():
    raise NameError("preprocessed_documents does not exist. Run the preprocessing cells before this OCR cell.")

ocr_results = []

for document in preprocessed_documents:
    filename = document["filename"]
    filename_stem = Path(filename).stem

    candidate_images = select_ocr_candidate_images(document)
    candidate_results = []

    for candidate_name, candidate_image in candidate_images.items():
        result = run_ocr_on_image(candidate_image)

        ocr_csv_path = OCR_DIR / f"{filename_stem}_{candidate_name}_ocr_words.csv"
        result["ocr_dataframe"].to_csv(ocr_csv_path, index=False)

        candidate_results.append({
            "candidate_name": candidate_name,
            "text": result["text"],
            "average_confidence": result["average_confidence"],
            "median_confidence": result["median_confidence"],
            "word_count": result["word_count"],
            "character_count": result["character_count"],
            "ocr_csv_path": str(ocr_csv_path)
        })

    best_candidate = max(
        candidate_results,
        key=lambda item: (
            item["average_confidence"],
            item["word_count"],
            item["character_count"]
        )
    )

    best_text_path = OCR_TEXT_DIR / f"{filename_stem}_best_ocr_text.txt"

    with open(best_text_path, "w", encoding="utf-8") as text_file:
        text_file.write(best_candidate["text"])

    ocr_results.append({
        "filename": filename,
        "candidate_results": candidate_results,
        "best_candidate_name": best_candidate["candidate_name"],
        "best_average_confidence": best_candidate["average_confidence"],
        "best_median_confidence": best_candidate["median_confidence"],
        "best_word_count": best_candidate["word_count"],
        "best_character_count": best_candidate["character_count"],
        "best_text": best_candidate["text"],
        "best_text_path": str(best_text_path)
    })

print("OCR completed for document images:")

for result in ocr_results:
    print(
        f"- {result['filename']}: "
        f"best={result['best_candidate_name']}, "
        f"avg_conf={result['best_average_confidence']:.2f}, "
        f"words={result['best_word_count']}"
    )


In [ ]:
# display OCR confidence comparison

ocr_summary_rows = []

for result in ocr_results:
    for candidate in result["candidate_results"]:
        ocr_summary_rows.append({
            "document": result["filename"],
            "ocr_source": candidate["candidate_name"],
            "average_confidence": round(candidate["average_confidence"], 2),
            "median_confidence": round(candidate["median_confidence"], 2),
            "word_count": candidate["word_count"],
            "character_count": candidate["character_count"],
            "selected_as_best": candidate["candidate_name"] == result["best_candidate_name"]
        })

ocr_summary_df = pd.DataFrame(ocr_summary_rows)

ocr_summary_csv_path = CSV_DIR / "ocr_confidence_summary.csv"
ocr_summary_df.to_csv(ocr_summary_csv_path, index=False)

ocr_summary_df


In [ ]:
# display best OCR text for the first document

example_ocr_result = ocr_results[0]

print("Document:", example_ocr_result["filename"])
print("Best OCR source:", example_ocr_result["best_candidate_name"])
print("Average confidence:", round(example_ocr_result["best_average_confidence"], 2))
print("Median confidence:", round(example_ocr_result["best_median_confidence"], 2))
print("Word count:", example_ocr_result["best_word_count"])
print()
print("Best OCR text preview:")
print("-" * 80)
print(example_ocr_result["best_text"][:2000])


# Multi-modal

## Stage 4 Multi-modal Plan

This section will later combine OCR, NLP, and computer vision outputs into one document-level result.

The OCR phase now provides:

1. best OCR text per document
2. OCR confidence by preprocessing method
3. OCR word-level CSV outputs
4. selected OCR source for each document

The next phases will use these OCR results for text preprocessing, feature extraction, section categorisation, segmentation, and text-image integration.

In [ ]:
# multi-modal development status for this commit

multimodal_development_status = {
    "ocr_text_extraction": "implemented",
    "ocr_confidence_comparison": "implemented",
    "ocr_word_csv_export": "implemented",
    "ocr_to_layout_mapping": "planned",
    "section_categorisation": "planned",
    "table_text_extraction": "planned",
    "diagram_reference_detection": "planned",
    "visual_feature_summary": "planned",
    "json_report_generation": "planned",
    "csv_summary_generation": "planned",
    "annotated_image_generation": "planned"
}

multimodal_development_status

# Final Output

In [ ]:
# final output summary for this development stage

# stage_2_summary = {
#     "stage": "document image loading",
#     "template_followed": True,
#     "input_documents_found": len(input_documents),
#     "successfully_loaded_documents": len(loaded_documents),
#     "input_document_names": [document["filename"] for document in loaded_documents],
#     "next_stage": "add grayscale conversion, denoising, and threshold preprocessing"
# }
#print(json.dumps(stage_2_summary, indent=4))

# stage_3_summary = {
#     "stage": "OpenCV image preprocessing and enhancement",
#     "template_followed": True,
#     "input_documents_found": len(input_documents),
#     "successfully_loaded_documents": len(loaded_documents),
#     "preprocessed_documents": len(preprocessed_documents),
#     "preprocessing_steps": [
#         "grayscale conversion",
#         "median blur denoising",
#         "CLAHE contrast enhancement",
#         "Otsu thresholding",
#         "adaptive thresholding",
#         "clean adaptive thresholding",
#         "Canny edge detection"
#     ],
#     "preprocessed_output_directory": str(PREPROCESSED_DIR),
#     "next_stage": "add Tesseract OCR extraction and OCR confidence reporting"
# }

# print(json.dumps(stage_3_summary, indent=4))

stage_4_summary = {
    "stage": "Tesseract OCR extraction and confidence comparison",
    "template_followed": True,
    "input_documents_found": len(input_documents),
    "successfully_loaded_documents": len(loaded_documents),
    "preprocessed_documents": len(preprocessed_documents),
    "ocr_documents_processed": len(ocr_results),
    "ocr_candidate_sources": [
        "gray",
        "denoised",
        "enhanced",
        "adaptive_threshold_clean"
    ],
    "ocr_summary_csv": str(ocr_summary_csv_path),
    "ocr_output_directory": str(OCR_DIR),
    "ocr_text_output_directory": str(OCR_TEXT_DIR),
    "next_stage": "add OCR text preprocessing with tokenisation, stopword removal, stemming, and lemmatisation"
}

print(json.dumps(stage_4_summary, indent=4))

In [ ]:
# development notes
# major commit 3 adds the first OpenCV image preprocessing stage
# OCR is tested against several preprocessed image versions instead of assuming one version is best
# the notebook compares Tesseract confidence scores and selects the best OCR result per document
# OCR word-level CSV files are saved to outputs/ocr_results
# best OCR text files are saved to outputs/ocr_text
# NLP preprocessing is intentionally left for the next commit